**Author**: Felipe Matheus
**Purpose**: Experiment launcher for the annealing **tensile strength (UTS)** surrogate.

Same architecture as `run_experiments.ipynb` (IACS): all pipeline logic lives
in `src/modeling/Experiments.py`, which is process-agnostic — the SAME
`ExperimentRunner` is reused; only the `ExperimentConfig` changes (target,
features, physical bounds). No new class needed.

Results layout: `models/annealing_tensile_strength/experiments/`
(`experiments_log.csv` + one folder per run).

# 1. Setup

In [ ]:
import logging
import os
import sys

import numpy as np
import pandas as pd

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.modeling.Modeling import Modeling
from src.modeling.Evaluation import Evaluation
from src.modeling.Experiments import ExperimentConfig, ExperimentRunner

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

%load_ext autoreload
%autoreload 2

proc = Processing()
feng = FeatureEngineering()
modl = Modeling()
evla = Evaluation()
runr = ExperimentRunner(modl, evla, models_root="../../models")

# 2. Data (same preparation as annealing_uts.ipynb, run once)

In [ ]:
PATH_DATA_RAW = "../../data/raw"
FILE_NAME = "dataset_annealing_tensile-strength.csv"

TARGET = "tensile_strength_final"
ALL_FEATURES = ["purity", "initial_diameter", "tensile_strength", "temperature", "time"]

df_raw = pd.read_csv(os.path.join(PATH_DATA_RAW, FILE_NAME))
df_float = proc.df_to_float(
    df_raw, drop_cols=["DOI", "is_Cu"], ignore_columns=["material"]
)
df_labeled = feng.label_element(df_float).drop_duplicates()
df_with_masks = feng.add_ratio_mask_column(
    feng.add_ratio_mask_column(df_labeled, "grain_size"), "tensile_strength",
)
df = (
    df_with_masks[ALL_FEATURES + [TARGET]]
    .dropna(subset=[TARGET])
    .reset_index(drop=True)
)

# No essay rows yet for UTS -> no is_essay column. The runr detects this
# and applies uniform weights (weight_on_essay_rows must stay 1.0).
print(f"Dataset: {df.shape}")

# Validation set: none held-out yet. When UTS essays arrive, build df_val
# from them (with the same columns) and pass df_val=df_val below.
df_val = None
df.head()

# 3. Base config

In [ ]:
base = ExperimentConfig(
    process="annealing_tensile_strength",
    tag="uts-v1",
    target=TARGET,
    features=tuple(ALL_FEATURES),
    # UTS has no 106-like ceiling; only positivity, far from observed values.
    y_max=None,
    y_min=0.0,
)
print(base.run_id)

# 4. Single run (sanity check before any grid)

Run the base config alone first; then repeat 2-3x with `tag="uts-v1-rep2"`
etc. to measure run-to-run noise (the floor below which grid differences
mean nothing).

In [ ]:
result = runr.run_experiment(df, base, df_val=df_val)
result["artifacts"]["metrics"]

# 5. Grid

In [ ]:
grid = {
    "time_limit_a": [60, 120],
    "num_bag_folds_a": [5, 10],
}
log = runr.run_grid(df, base, grid, df_val=df_val)
log

# 6. Inspect results

In [ ]:
log = runr.load_log(base)

view_cols = [
    "run_id", "cfg_time_limit_a", "cfg_num_bag_folds_a", "cfg_features",
    "rmse", "mae", "cov_0.9", "val_rmse", "val_mae", "val_cov_0.9",
    "c_opt", "pct_truncated_aleat", "mean_sigma_epist", "mean_sigma_aleat",
    "elapsed_s",
]
log[[c for c in view_cols if c in log.columns]].sort_values("rmse")

# 7. Load a winner

In [ ]:
import pickle
from pathlib import Path
from autogluon.tabular import TabularPredictor

RUN_ID = log.sort_values("rmse").iloc[0]["run_id"]
run_dir = Path("../../models/annealing_tensile_strength/experiments") / RUN_ID

with open(run_dir / "artifacts.pkl", "rb") as f:
    art = pickle.load(f)
predictor_a = TabularPredictor.load(str(run_dir / "model_a"))
predictor_b = TabularPredictor.load(str(run_dir / "model_b"))

print(RUN_ID)
art["calibration_after"]